In [1]:
afc_countries = {
    1: "Afghanistan", 12: "Australia", 16: "Bahrain", 17: "Bangladesh",
    24: "Bhutan", 30: "Brunei", 34: "Cambodia", 42: "China",
    76: "Guam", 83: "Hong_Kong", 86: "India", 87: "Indonesia",
    88: "Iran", 89: "Iraq", 94: "Japan", 95: "Jordan",
    99: "Kuwait", 100: "Kyrgyzstan", 101: "Laos", 103: "Lebanon",
    110: "Macau", 114: "Malaysia", 115: "Maldives", 123: "Mongolia",
    128: "Nepal", 136: "North_Korea", 139: "Oman", 140: "Pakistan",
    141: "Palestine", 146: "Philippines", 150: "Qatar", 161: "Saudi_Arabia",
    167: "Singapore", 173: "South_Korea", 175: "Sri_Lanka", 181: "Syria",
    183: "Taiwan", 184: "Tajikistan", 186: "Thailand", 193: "Turkmenistan",
    197: "United_Arab_Emirates", 201: "Uzbekistan", 204: "Vietnam", 206: "Yemen",
    214: "Northern_Mariana_Islands", 229: "East_Timor", 230: "Myanmar"
}

In [2]:
# Run scraper to test on Vietnam 2025 page

import re
from typing import Any, Dict, List, Optional
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup


def scrape_country_year_info(country_id: int, year: int, country_slug: str) -> Dict[str, Any]:
    """Scrape average height, average age, and match report IDs from national-football-teams."""
    base_url = "https://national-football-teams.com"
    page_url = f"{base_url}/country/{country_id}/{year}/{country_slug}.html"

    response = requests.get(page_url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    def find_value_after_label(label_pattern: str) -> Optional[str]:
        # Find the <strong> label, then read value from its parent row.
        strong = soup.find("strong", string=re.compile(label_pattern, re.IGNORECASE))
        if not strong:
            return None
        row = strong.find_parent("div", class_="row")
        if not row:
            return None
        cols = row.find_all("div", class_=re.compile(r"\bcol-\d+\b"))
        if len(cols) >= 2:
            return cols[1].get_text(" ", strip=True) or None
        return None

    avg_height = find_value_after_label(r"Average\s+height\s+in\s+\d{4}")
    avg_age = find_value_after_label(r"Average\s+age\s+in\s+\d{4}")

    match_ids: List[int] = []
    seen = set()
    for a_tag in soup.select("a[href*='/matches/report/']"):
        href = a_tag.get("href", "")
        m = re.search(r"/matches/report/(\d+)/", href)
        if m:
            match_id = int(m.group(1))
            if match_id not in seen:
                seen.add(match_id)
                match_ids.append(match_id)

    return {
        "country_id": country_id,
        "country_slug": country_slug,
        "year": year,
        "page_url": page_url,
        "average_height": avg_height,
        "average_age": avg_age,
        "match_ids": match_ids,
        "match_count": len(match_ids),
    }


result = scrape_country_year_info(204, 2025, "Vietnam")
result

{'country_id': 204,
 'country_slug': 'Vietnam',
 'year': 2025,
 'page_url': 'https://national-football-teams.com/country/204/2025/Vietnam.html',
 'average_height': '1.77m',
 'average_age': '26.1',
 'match_ids': [40420, 40421, 41185, 41131, 41697, 42366, 42481, 42782],
 'match_count': 8}

In [3]:
# Run a full scraper from https://national-football-teams.com

import asyncio
import random
import re
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Dict, List, Tuple

import httpx
import pandas as pd
from bs4 import BeautifulSoup


BASE_URL = "https://national-football-teams.com"


@dataclass
class ScrapeStop(Exception):
    """Signal to stop the whole scraping job due to critical HTTP/network errors."""

    message: str


def extract_country_year_info_from_html(html: str, page_url: str, country_id: int, country_slug: str, year: int):
    """Parse average height, average age, and unique match refs from one country-year HTML page."""
    soup = BeautifulSoup(html, "html.parser")

    def find_value_after_label(label_pattern: str):
        strong = soup.find("strong", string=re.compile(label_pattern, re.IGNORECASE))
        if not strong:
            return None
        row = strong.find_parent("div", class_="row")
        if not row:
            return None
        cols = row.find_all("div", class_=re.compile(r"\bcol-\d+\b"))
        if len(cols) >= 2:
            value = cols[1].get_text(" ", strip=True)
            return value or None
        return None

    avg_height = find_value_after_label(r"Average\s+height\s+in\s+\d{4}")
    avg_age = find_value_after_label(r"Average\s+age\s+in\s+\d{4}")

    # Keep page-level refs unique while preserving order.
    # Target format: 44212/Myanmar_Afghanistan.html
    page_match_refs: List[str] = []
    seen_page_refs = set()
    for a_tag in soup.select("a[href*='/matches/report/']"):
        href = str(a_tag.get("href", "")).split("?", 1)[0].strip()
        m = re.search(r"/matches/report/(\d+/[^/]+\.html)$", href)
        if not m:
            continue
        match_ref = m.group(1)
        if match_ref not in seen_page_refs:
            seen_page_refs.add(match_ref)
            page_match_refs.append(match_ref)

    # Treat pages with no key data and no match links as effectively empty for this task.
    has_data = bool(avg_height or avg_age or page_match_refs)

    row = {
        "country_id": country_id,
        "country_slug": country_slug,
        "year": year,
        "page_url": page_url,
        "average_height": avg_height,
        "average_age": avg_age,
        "match_count": len(page_match_refs),
        "match_ids": page_match_refs,
        "scraped_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }
    return row, has_data, page_match_refs


async def _fetch_country_year_page(
    client: httpx.AsyncClient,
    page_url: str,
    request_timeout: int,
):
    try:
        resp = await client.get(page_url, timeout=request_timeout)
        return resp, None
    except httpx.RequestError as exc:
        return None, str(exc)


async def scrape_all_afc_country_years(
    countries: Dict[int, str],
    start_year: int = 1995,
    end_year: int = 2026,
    sleep_range: Tuple[float, float] = (1, 2),
    long_break_every: int = 50,
    long_break_seconds: int = 20,
    stop_on_http_errors: bool = True,
    request_timeout: int = 30,
    concurrency: int = 8,
    user_agent: str = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
):
    """
    Async bulk scrape all country/year pages.

    Rules:
    - 404 => skip this country/year
    - no data on page => skip storing row
    - 403 or non-404 HTTP/network errors => stop whole run
    - sleep randomly between batches to reduce ban risk
    - take a long break every N requests
    """
    if sleep_range[0] < 0 or sleep_range[1] < sleep_range[0]:
        raise ValueError("sleep_range must be (min_seconds, max_seconds) with 0 <= min <= max")
    if long_break_every <= 0:
        raise ValueError("long_break_every must be > 0")
    if long_break_seconds < 0:
        raise ValueError("long_break_seconds must be >= 0")
    if concurrency <= 0:
        raise ValueError("concurrency must be > 0")

    records = []
    global_match_ids = set()  # Global deduplicated refs like 44212/Team_A_B.html.
    match_ids_by_country_year = {}  # (country_id, year) -> set(match_refs)
    stats = {
        "total_requests": 0,
        "stored_rows": 0,
        "skipped_404": 0,
        "skipped_empty": 0,
        "long_breaks_taken": 0,
        "stopped": False,
        "stop_reason": None,
    }

    tasks = []
    for country_id, country_slug in countries.items():
        for year in range(start_year, end_year + 1):
            tasks.append((country_id, country_slug, year))

    async with httpx.AsyncClient(headers={"User-Agent": user_agent}, follow_redirects=True) as client:
        stop_now = False

        for start_idx in range(0, len(tasks), concurrency):
            if stop_now:
                break

            batch = tasks[start_idx : start_idx + concurrency]
            coroutines = []
            page_urls = []

            for country_id, country_slug, year in batch:
                page_url = f"{BASE_URL}/country/{country_id}/{year}/{country_slug}.html"
                page_urls.append((country_id, country_slug, year, page_url))
                coroutines.append(
                    _fetch_country_year_page(
                        client=client,
                        page_url=page_url,
                        request_timeout=request_timeout,
                    )
                )

            results = await asyncio.gather(*coroutines)

            for (country_id, country_slug, year, page_url), (resp, request_err) in zip(page_urls, results):
                stats["total_requests"] += 1

                if request_err is not None:
                    stats["stopped"] = True
                    stats["stop_reason"] = f"Network/requests error at {page_url}: {request_err}"
                    stop_now = True
                    break

                if resp is None:
                    stats["stopped"] = True
                    stats["stop_reason"] = f"Unknown response error at {page_url}"
                    stop_now = True
                    break

                if resp.status_code == 404:
                    stats["skipped_404"] += 1
                    continue

                if resp.status_code == 403:
                    stats["stopped"] = True
                    stats["stop_reason"] = f"HTTP 403 at {page_url}. Server denied access."
                    stop_now = True
                    break

                if resp.status_code != 200:
                    if stop_on_http_errors:
                        stats["stopped"] = True
                        stats["stop_reason"] = f"HTTP {resp.status_code} at {page_url}"
                        stop_now = True
                        break
                    continue

                row, has_data, page_match_refs = extract_country_year_info_from_html(
                    html=resp.text,
                    page_url=page_url,
                    country_id=country_id,
                    country_slug=country_slug,
                    year=year,
                )

                if has_data:
                    records.append(row)
                    stats["stored_rows"] += 1

                    key = (country_id, year)
                    match_set = set(page_match_refs)
                    match_ids_by_country_year[key] = match_set
                    global_match_ids.update(match_set)
                else:
                    stats["skipped_empty"] += 1

                if stats["total_requests"] % long_break_every == 0:
                    stats["long_breaks_taken"] += 1
                    print(
                        f"[PAUSE] Completed {stats['total_requests']} requests. "
                        f"Sleeping {long_break_seconds}s for cooldown..."
                    )
                    await asyncio.sleep(long_break_seconds)

            if stop_now:
                break

            # Random short pause between batches.
            await asyncio.sleep(random.uniform(*sleep_range))

    if stats["stopped"]:
        print(f"[STOP] {stats['stop_reason']}")

    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values(["country_id", "year"]).reset_index(drop=True)

    result = {
        "df_country_year": df,
        "global_match_ids": global_match_ids,
        "match_ids_by_country_year": match_ids_by_country_year,
        "stats": stats,
    }
    return result


RUN_FULL_SCRAPE = False

if RUN_FULL_SCRAPE:
    bulk_result = await scrape_all_afc_country_years(
        countries=afc_countries,
        start_year=1995,
        end_year=2026,
        sleep_range=(1, 2),
        long_break_every=50,
        long_break_seconds=20,
        stop_on_http_errors=True,
        request_timeout=30,
        concurrency=8,
    )

    df_country_year = bulk_result["df_country_year"]
    global_match_ids = bulk_result["global_match_ids"]
    match_ids_by_country_year = bulk_result["match_ids_by_country_year"]
    stats = bulk_result["stats"]

    print("Stats:", stats)
    print("Rows in df_country_year:", len(df_country_year))
    print("Unique global match refs:", len(global_match_ids))
    display(df_country_year.head())
else:
    print("Set RUN_FULL_SCRAPE = True to start bulk scraping.")

Set RUN_FULL_SCRAPE = True to start bulk scraping.


In [ ]:
import ast
from pathlib import Path

import pandas as pd


EXPORT_DIR = Path("../downloaded_files/scrape_exports")
COUNTRY_YEAR_CSV = EXPORT_DIR / "afc_country_year_data.csv"


def _parse_match_refs(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, (list, tuple, set)):
        return [str(x).strip() for x in value if str(x).strip()]
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, (list, tuple, set)):
                return [str(x).strip() for x in parsed if str(x).strip()]
        except (ValueError, SyntaxError):
            pass
        return [x.strip() for x in text.split(",") if x.strip()]
    return []


if "df_country_year" not in globals():
    if COUNTRY_YEAR_CSV.exists():
        df_country_year = pd.read_csv(COUNTRY_YEAR_CSV)
        print(f"Loaded df_country_year from {COUNTRY_YEAR_CSV} ({len(df_country_year)} rows).")
    else:
        df_country_year = pd.DataFrame()
        print(f"{COUNTRY_YEAR_CSV} not found. Initialized empty df_country_year.")

# Optional alias if you prefer this variable name.
afc_country_year = df_country_year

if "global_match_ids" not in globals() or "match_ids_by_country_year" not in globals():
    global_match_ids = set()
    match_ids_by_country_year = {}
    if not df_country_year.empty and {"country_id", "year", "match_ids"}.issubset(df_country_year.columns):
        for row in df_country_year[["country_id", "year", "match_ids"]].itertuples(index=False):
            key = (int(row.country_id), int(row.year))
            refs = set(_parse_match_refs(row.match_ids))
            match_ids_by_country_year[key] = refs
            global_match_ids.update(refs)
    print(
        "Initialized missing match-id state:",
        f"country_year_keys={len(match_ids_by_country_year)},",
        f"global_match_refs={len(global_match_ids)}",
    )

if "stats" not in globals() or not isinstance(stats, dict):
    stats = {
        "total_requests": 0,
        "stored_rows": len(df_country_year),
        "skipped_404": 0,
        "skipped_empty": 0,
        "long_breaks_taken": 0,
        "stopped": False,
        "stop_reason": None,
    }
    print("Initialized default stats for resume mode.")


Loaded df_country_year from downloaded_files/scrape_exports/afc_country_year_data.csv (1504 rows).
Initialized missing match-id state: country_year_keys=1504, global_match_refs=7587
Initialized default stats for resume mode.


In [5]:
import asyncio
import random
import re
from typing import Dict, Tuple

import httpx
import pandas as pd


def build_country_year_tasks(countries: Dict[int, str], start_year: int, end_year: int):
    tasks = []
    for country_id, country_slug in countries.items():
        for year in range(start_year, end_year + 1):
            tasks.append((country_id, year, country_slug))
    return tasks


def parse_failed_country_year_from_stats(previous_stats: dict):
    """Extract failed URL + (country_id, year) from a stop reason like: HTTP 503 at <url>."""
    if not previous_stats:
        return None
    reason = previous_stats.get("stop_reason") or ""
    url_match = re.search(r"https?://\S+", reason)
    if not url_match:
        return None
    failed_url = url_match.group(0).rstrip(".")
    page_match = re.search(r"/country/(\d+)/(\d+)/([^/]+)\.html", failed_url)
    if not page_match:
        return None
    return {
        "failed_url": failed_url,
        "country_id": int(page_match.group(1)),
        "year": int(page_match.group(2)),
        "country_slug": page_match.group(3),
    }


async def _fetch_resume_country_year_page(
    client: httpx.AsyncClient,
    page_url: str,
    request_timeout: int,
):
    try:
        resp = await client.get(page_url, timeout=request_timeout)
        return resp, None
    except httpx.RequestError as exc:
        return None, str(exc)


async def resume_afc_scrape(
    countries: Dict[int, str],
    existing_df: pd.DataFrame,
    existing_global_match_ids: set,
    existing_match_ids_by_country_year: dict,
    previous_stats: dict,
    start_year: int = 1995,
    end_year: int = 2026,
    sleep_range: Tuple[float, float] = (1, 2),
    long_break_every: int = 50,
    long_break_seconds: int = 20,
    stop_on_http_errors: bool = True,
    request_timeout: int = 30,
    resume_from_last_failed_url: bool = True,
    concurrency: int = 8,
    user_agent: str = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
):
    """Resume scraping using existing in-memory results, retrying the last failed page first (async/httpx)."""
    if sleep_range[0] < 0 or sleep_range[1] < sleep_range[0]:
        raise ValueError("sleep_range must be (min_seconds, max_seconds) with 0 <= min <= max")
    if long_break_every <= 0:
        raise ValueError("long_break_every must be > 0")
    if long_break_seconds < 0:
        raise ValueError("long_break_seconds must be >= 0")
    if concurrency <= 0:
        raise ValueError("concurrency must be > 0")

    tasks = build_country_year_tasks(countries, start_year, end_year)
    task_index = {(cid, year): idx for idx, (cid, year, _slug) in enumerate(tasks)}

    failed_info = parse_failed_country_year_from_stats(previous_stats)
    if resume_from_last_failed_url and failed_info:
        start_idx = task_index.get((failed_info["country_id"], failed_info["year"]), 0)
    else:
        start_idx = 0

    records = existing_df.to_dict("records") if existing_df is not None and not existing_df.empty else []
    global_match_ids = set(existing_global_match_ids or set())
    match_ids_by_country_year = {
        key: set(value)
        for key, value in (existing_match_ids_by_country_year or {}).items()
    }

    processed_keys = set(match_ids_by_country_year.keys())
    if existing_df is not None and not existing_df.empty:
        processed_keys.update(
            (int(row.country_id), int(row.year))
            for row in existing_df[["country_id", "year"]].itertuples(index=False)
        )

    pending_tasks = []
    for idx in range(start_idx, len(tasks)):
        country_id, year, country_slug = tasks[idx]
        key = (country_id, year)
        if key in processed_keys:
            continue
        pending_tasks.append((country_id, year, country_slug))

    resume_stats = {
        "total_requests": 0,
        "stored_rows": 0,
        "skipped_404": 0,
        "skipped_empty": 0,
        "long_breaks_taken": 0,
        "stopped": False,
        "stop_reason": None,
    }

    async with httpx.AsyncClient(headers={"User-Agent": user_agent}, follow_redirects=True) as client:
        stop_now = False

        for start in range(0, len(pending_tasks), concurrency):
            if stop_now:
                break

            batch = pending_tasks[start : start + concurrency]
            page_info = []
            coroutines = []

            for country_id, year, country_slug in batch:
                page_url = f"{BASE_URL}/country/{country_id}/{year}/{country_slug}.html"
                page_info.append((country_id, year, country_slug, page_url))
                coroutines.append(
                    _fetch_resume_country_year_page(
                        client=client,
                        page_url=page_url,
                        request_timeout=request_timeout,
                    )
                )

            results = await asyncio.gather(*coroutines)

            for (country_id, year, country_slug, page_url), (resp, request_err) in zip(page_info, results):
                key = (country_id, year)
                resume_stats["total_requests"] += 1

                if request_err is not None:
                    resume_stats["stopped"] = True
                    resume_stats["stop_reason"] = f"Network/requests error at {page_url}: {request_err}"
                    stop_now = True
                    break

                if resp is None:
                    resume_stats["stopped"] = True
                    resume_stats["stop_reason"] = f"Unknown response error at {page_url}"
                    stop_now = True
                    break

                if resp.status_code == 404:
                    resume_stats["skipped_404"] += 1
                    continue

                if resp.status_code == 403:
                    resume_stats["stopped"] = True
                    resume_stats["stop_reason"] = f"HTTP 403 at {page_url}. Server denied access."
                    stop_now = True
                    break

                if resp.status_code != 200:
                    if stop_on_http_errors:
                        resume_stats["stopped"] = True
                        resume_stats["stop_reason"] = f"HTTP {resp.status_code} at {page_url}"
                        stop_now = True
                        break
                    continue

                row, has_data, page_match_ids = extract_country_year_info_from_html(
                    html=resp.text,
                    page_url=page_url,
                    country_id=country_id,
                    country_slug=country_slug,
                    year=year,
                )

                if has_data:
                    records.append(row)
                    processed_keys.add(key)
                    resume_stats["stored_rows"] += 1

                    page_set = set(page_match_ids)
                    match_ids_by_country_year[key] = page_set
                    global_match_ids.update(page_set)
                else:
                    resume_stats["skipped_empty"] += 1

                if resume_stats["total_requests"] % long_break_every == 0:
                    resume_stats["long_breaks_taken"] += 1
                    print(
                        f"[PAUSE] Resume completed {resume_stats['total_requests']} requests. "
                        f"Sleeping {long_break_seconds}s for cooldown..."
                    )
                    await asyncio.sleep(long_break_seconds)

            if stop_now:
                break

            await asyncio.sleep(random.uniform(*sleep_range))

    if resume_stats["stopped"]:
        print(f"[STOP] {resume_stats['stop_reason']}")

    updated_df = pd.DataFrame(records)
    if not updated_df.empty:
        updated_df = updated_df.drop_duplicates(subset=["country_id", "year"], keep="first")
        updated_df = updated_df.sort_values(["country_id", "year"]).reset_index(drop=True)

    merged_stats = dict(previous_stats or {})
    count_keys = ["total_requests", "stored_rows", "skipped_404", "skipped_empty", "long_breaks_taken"]
    for key in count_keys:
        merged_stats[key] = merged_stats.get(key, 0) + resume_stats[key]
    merged_stats["stopped"] = resume_stats["stopped"]
    merged_stats["stop_reason"] = resume_stats["stop_reason"]
    merged_stats["resume_requests"] = resume_stats["total_requests"]
    merged_stats["resume_started_from_failed_url"] = bool(failed_info)

    return {
        "df_country_year": updated_df,
        "global_match_ids": global_match_ids,
        "match_ids_by_country_year": match_ids_by_country_year,
        "stats": merged_stats,
        "resume_stats": resume_stats,
        "resume_failed_info": failed_info,
    }


RUN_RESUME_SCRAPE = False

if RUN_RESUME_SCRAPE:
    resumed_result = await resume_afc_scrape(
        countries=afc_countries,
        existing_df=df_country_year,
        existing_global_match_ids=global_match_ids,
        existing_match_ids_by_country_year=match_ids_by_country_year,
        previous_stats=stats,
        start_year=1995,
        end_year=2026,
        sleep_range=(1, 2),
        long_break_every=50,
        long_break_seconds=20,
        stop_on_http_errors=True,
        request_timeout=30,
        resume_from_last_failed_url=True,
        concurrency=8,
    )

    bulk_result = resumed_result
    df_country_year = resumed_result["df_country_year"]
    global_match_ids = resumed_result["global_match_ids"]
    match_ids_by_country_year = resumed_result["match_ids_by_country_year"]
    stats = resumed_result["stats"]

    print("Resume stats:", resumed_result["resume_stats"])
    print("Merged stats:", stats)
    print("Rows in df_country_year:", len(df_country_year))
    print("Unique global match refs:", len(global_match_ids))
    display(df_country_year.tail())
else:
    print("Set RUN_RESUME_SCRAPE = True in Cell 4 to continue from the last failed URL.")

Set RUN_RESUME_SCRAPE = True in Cell 4 to continue from the last failed URL.


In [6]:
# from pathlib import Path
# import json

# # Save all current scrape outputs so the scraping step does not need to be rerun.
# output_dir = Path("downloaded_files/scrape_exports")
# output_dir.mkdir(parents=True, exist_ok=True)

# # 1) Main country-year table
# df_csv_path = output_dir / "afc_country_year_data.csv"
# df_country_year.to_csv(df_csv_path, index=False, encoding="utf-8")

# # 2) Global unique match IDs (one ID per row for easy reuse)
# global_ids_csv_path = output_dir / "afc_global_match_ids.csv"
# global_ids_df = pd.DataFrame({"match_id": sorted(global_match_ids)})
# global_ids_df.to_csv(global_ids_csv_path, index=False, encoding="utf-8")

# # 3) Country-year -> match IDs mapping in CSV form (normalized rows)
# mapping_rows = []
# for (country_id, year), id_set in sorted(match_ids_by_country_year.items()):
#     for match_id in sorted(id_set):
#         mapping_rows.append({
#             "country_id": country_id,
#             "year": year,
#             "match_id": match_id,
#         })

# mapping_csv_path = output_dir / "afc_match_ids_by_country_year.csv"
# pd.DataFrame(mapping_rows).to_csv(mapping_csv_path, index=False, encoding="utf-8")

# # Optional JSON copy for direct dictionary reload later
# mapping_json_path = output_dir / "afc_match_ids_by_country_year.json"
# with mapping_json_path.open("w", encoding="utf-8") as f:
#     json.dump(
#         {f"{country_id}_{year}": sorted(list(id_set)) for (country_id, year), id_set in match_ids_by_country_year.items()},
#         f,
#         ensure_ascii=False,
#         indent=2,
#     )

# print("Saved files:")
# print("-", df_csv_path)
# print("-", global_ids_csv_path)
# print("-", mapping_csv_path)
# print("-", mapping_json_path)
# print("Rows in main CSV:", len(df_country_year))
# print("Unique global match IDs:", len(global_match_ids))

In [7]:
import re
from typing import Dict, List, Optional

import requests
from bs4 import BeautifulSoup


def _clean_name(given: Optional[str], family: Optional[str], fallback: str = "") -> str:
    given = (given or "").strip()
    family = (family or "").strip()
    full = f"{given} {family}".strip()
    return full if full else fallback.strip()


def _team_name_from_flag_src(src: str) -> str:
    filename = src.rsplit("/", 1)[-1]
    stem = filename.split("-", 1)[0]
    team = re.sub(r"_\d+$", "", stem)
    return team.replace("_", " ").strip()


def scrape_match_events(soup: BeautifulSoup) -> Dict[str, List[Dict[str, str]]]:
    events = {"goals": [], "cards": []}
    for p in soup.find_all("p"):
        text = p.get_text(" ", strip=True)
        links = p.find_all("a")
        if "scored a goal" in text or "scored an own goal" in text:
            scorer = links[0].get_text(strip=True) if links else None
            assist = None
            is_own_goal = "own goal" in text
            if "assist by" in text and len(links) > 1:
                assist = links[1].get_text(strip=True)
            if scorer:
                events["goals"].append({"scorer": scorer, "assist": assist, "own_goal": is_own_goal})
        elif "cautioned with" in text:
            player = links[0].get_text(strip=True) if links else None
            if not player: continue
            if "second yellow card" in text:
                card_type = "Second yellow"
            elif "red card" in text:
                card_type = "Red"
            else:
                card_type = "Yellow"
            events["cards"].append({"player": player, "card_type": card_type})
    return events


def scrape_match_lineups_and_coaches(match_report_url: str, timeout: int = 30) -> Dict[str, object]:
    """Scrape team lineups, coaches, date, goals, and cards."""
    resp = requests.get(match_report_url, timeout=timeout)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    
    date_str = None
    h1 = soup.find("h1")
    if h1:
        small = h1.find("small")
        if small:
            date_str = small.get_text(strip=True)

    teams: List[Dict[str, object]] = []

    for col in soup.select("div.col-md-4"):
        starting_heading = col.find(["h5", "h6"], string=re.compile(r"Starting\s+Line-?Up", re.IGNORECASE))
        if not starting_heading:
            continue

        team_name = None
        flag_img = col.select_one("img.icon-featured")
        if flag_img and flag_img.get("src"):
            team_name = _team_name_from_flag_src(flag_img["src"])

        lineup_players: List[str] = []
        starting_players_container = starting_heading.find_parent("div", class_=re.compile(r"heading"))
        if starting_players_container:
            lineup_box = starting_players_container.find_next("div", class_=re.compile(r"players"))
            if lineup_box:
                for player_row in lineup_box.select("div.row.player"):
                    given_span = player_row.select_one("span[itemprop='givenName']")
                    family_span = player_row.select_one("span[itemprop='familyName']")
                    anchor = player_row.select_one("a[itemprop='url']")

                    player_name = _clean_name(
                        given_span.get_text(strip=True) if given_span else None,
                        family_span.get_text(strip=True) if family_span else None,
                        anchor.get_text(" ", strip=True) if anchor else "",
                    )
                    if player_name:
                        lineup_players.append(player_name)

        coach_name = None
        coach_block = col.select_one("div.coach[itemprop='coach']")
        if coach_block:
            given = coach_block.select_one("span[itemprop='givenName']")
            family = coach_block.select_one("span[itemprop='familyName']")
            coach_name = _clean_name(
                given.get_text(strip=True) if given else None,
                family.get_text(strip=True) if family else None,
            )

        teams.append({
            "team": team_name,
            "starting_lineup": lineup_players,
            "coach": coach_name,
        })

    events = scrape_match_events(soup)

    return {
        "match_report_url": match_report_url,
        "date": date_str,
        "teams": teams,
        "events": events,
    }

match_url = "https://national-football-teams.com/matches/report/37475/Afghanistan_Mongolia.html"
lineup_data = scrape_match_lineups_and_coaches(match_url)
lineup_data

{'match_report_url': 'https://national-football-teams.com/matches/report/37475/Afghanistan_Mongolia.html',
 'date': '2023-10-12',
 'teams': [{'team': 'Afghanistan',
   'starting_lineup': ['Faisal Hamidi',
    'Mahboob Hanifi',
    'Najim Haidary',
    'Mosawer Ahadi',
    'Omid Popalzay',
    'Farshad Noor',
    'Omid Musawi',
    'Noor Husin',
    'Rahmat Akbari',
    'Faysal Shayesteh',
    'Omran Haydary'],
   'coach': 'Abdullah Al-Mutairi'},
  {'team': 'Mongolia',
   'starting_lineup': ['Ariunbold Batsaikhan',
    'Mönkh-Orgil Orkhon',
    'Filip Andersen Chinzorig',
    'Bat-Orgil Gerelt-Old',
    'Amgalanbat Batbaatar',
    'Tsend-Ayush Khürelbaatar',
    'Batmönkh Baljinnyam',
    'Baasanjav Tserenbat',
    'Mijiddorj Oyunbaatar',
    'Ganbold Ganbayar',
    'Dölgöön Amaraa'],
   'coach': 'Ichiro Otsuka'}],
 'events': {'goals': [{'scorer': 'Jabar Sharza',
    'assist': 'Mosawer Ahadi',
    'own_goal': False}],
  'cards': [{'player': 'Filip Andersen Chinzorig', 'card_type': 'Yell

In [8]:
sample_match_ref = None
if "match_ids_by_country_year" not in globals():
    raise ValueError("Run Cell 4 first so match_ids_by_country_year is available.")

for match_refs in match_ids_by_country_year.values():
    if match_refs:
        sample_match_ref = sorted(str(match_ref) for match_ref in match_refs)[0]
        break

if sample_match_ref is None:
    raise ValueError("No match refs were found in match_ids_by_country_year.")

sample_match_url = f"{BASE_URL}/matches/report/{sample_match_ref}"
print("Testing one match:", sample_match_url)
single_game_preview = scrape_match_lineups_and_coaches(sample_match_url)
single_game_preview

Testing one match: https://national-football-teams.com/matches/report/1411/Kyrgyzstan_Afghanistan.html


{'match_report_url': 'https://national-football-teams.com/matches/report/1411/Kyrgyzstan_Afghanistan.html',
 'date': '2003-03-16',
 'teams': [{'team': 'Kyrgyzstan',
   'starting_lineup': ['Zamirbek Jumagulov',
    'Vyacheslav Pryanishnikov',
    'Vyacheslav Amin',
    'Vladimir Salo',
    'Zakir Jalilov',
    'Kanat Sardarov',
    'Atabek Berdaly uulu',
    'Dmitriy Krokhmal',
    'Islam Kurmanbayev',
    'Nurlan Rajabaliev',
    'Murat Jumakeyev'],
   'coach': 'Nematjan Zakirov'},
  {'team': 'Afghanistan',
   'starting_lineup': ['Raza Razai',
    'Sayed Tahir Shah Mosawe',
    'Najib Naderi',
    'Omar Nazar',
    'Ahmed Zia Azimi',
    'Abdul Salem Jamshid',
    'Bashir Ahmad Saadat',
    'Najiballah Zarimi',
    'Qais Khedri',
    'Ahmad Rahil Fourmoli',
    'Enaltat'],
   'coach': 'Mir Ali Asghar Akbarzada'}],
 'events': {'goals': [{'scorer': 'Farid Ahmadi',
    'assist': None,
    'own_goal': False},
   {'scorer': 'Zamirbek Jumagulov', 'assist': None, 'own_goal': False},
   {'scor

In [9]:
import asyncio
import random
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple

import httpx
import pandas as pd
from bs4 import BeautifulSoup


BASE_URL = "https://national-football-teams.com"


def _clean_name(given: Optional[str], family: Optional[str], fallback: str = "") -> str:
    given = (given or "").strip()
    family = (family or "").strip()
    full = f"{given} {family}".strip()
    return full if full else fallback.strip()


def _team_name_from_flag_src(src: str) -> str:
    # Example src: .../flag/Afghanistan_1-58948ed43c136.png
    filename = src.rsplit("/", 1)[-1]
    stem = filename.split("-", 1)[0]
    team = re.sub(r"_\d+$", "", stem)
    return team.replace("_", " ").strip()


def extract_match_lineups_and_coaches_from_html(match_ref: str, match_url: str, html: str) -> List[Dict[str, object]]:
    """Extract starting lineup players and coach per team from one match report HTML."""
    soup = BeautifulSoup(html, "html.parser")
    rows: List[Dict[str, object]] = []

    date_str = None
    h1 = soup.find("h1")
    if h1:
        small = h1.find("small")
        if small:
            date_str = small.get_text(strip=True)

    player_stats = {}
    def get_stats(name):
        if name not in player_stats:
            player_stats[name] = {"goals": 0, "assists": 0, "own_goals": 0, "yellow_cards": 0, "red_cards": 0}
        return player_stats[name]

    for p in soup.find_all("p"):
        text = p.get_text(" ", strip=True)
        links = p.find_all("a")
        if not links:
            continue
        if "scored a goal" in text or "scored an own goal" in text:
            scorer = links[0].get_text(strip=True)
            is_own = "own goal" in text
            if is_own:
                get_stats(scorer)["own_goals"] += 1
            else:
                get_stats(scorer)["goals"] += 1
            
            if "assist by" in text and len(links) > 1:
                assist = links[1].get_text(strip=True)
                get_stats(assist)["assists"] += 1
        elif "cautioned with" in text:
            player = links[0].get_text(strip=True)
            if "second yellow card" in text or "red card" in text:
                get_stats(player)["red_cards"] += 1
            else:
                get_stats(player)["yellow_cards"] += 1

    for col in soup.select("div.col-md-4"):
        starting_heading = col.find(["h5", "h6"], string=re.compile(r"Starting\s+Line-?Up", re.IGNORECASE))
        if not starting_heading:
            continue

        team_name = None
        flag_img = col.select_one("img.icon-featured")
        if flag_img and flag_img.get("src"):
            team_name = _team_name_from_flag_src(str(flag_img["src"]))

        coach_name = None
        coach_block = col.select_one("div.coach[itemprop='coach']")
        if coach_block:
            given = coach_block.select_one("span[itemprop='givenName']")
            family = coach_block.select_one("span[itemprop='familyName']")
            coach_name = _clean_name(
                given.get_text(strip=True) if given else None,
                family.get_text(strip=True) if family else None,
            )

        starting_players: List[str] = []
        heading_container = starting_heading.find_parent("div", class_=re.compile(r"heading"))
        if heading_container:
            players_box = heading_container.find_next("div", class_=re.compile(r"players"))
            if players_box:
                for player_row in players_box.select("div.row.player"):
                    given_span = player_row.select_one("span[itemprop='givenName']")
                    family_span = player_row.select_one("span[itemprop='familyName']")
                    anchor = player_row.select_one("a[itemprop='url']")

                    player_name = _clean_name(
                        given_span.get_text(strip=True) if given_span else None,
                        family_span.get_text(strip=True) if family_span else None,
                        anchor.get_text(" ", strip=True) if anchor else "",
                    )
                    if player_name:
                        starting_players.append(player_name)

        scraped_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
        for order, player_name in enumerate(starting_players, start=1):
            stats = player_stats.pop(player_name, {"goals": 0, "assists": 0, "own_goals": 0, "yellow_cards": 0, "red_cards": 0})
            rows.append(
                {
                    "match_id": match_ref,
                    "match_url": match_url,
                    "date": date_str,
                    "team": team_name,
                    "coach": coach_name,
                    "player_name": player_name,
                    "player_order": order,
                    "goals": stats["goals"],
                    "assists": stats["assists"],
                    "own_goals": stats["own_goals"],
                    "yellow_cards": stats["yellow_cards"],
                    "red_cards": stats["red_cards"],
                    "scraped_at_utc": scraped_at,
                }
            )

        # Keep one row even if no lineup players are found, so the match still counts as scraped.
        if not starting_players:
            rows.append(
                {
                    "match_id": match_ref,
                    "match_url": match_url,
                    "date": date_str,
                    "team": team_name,
                    "coach": coach_name,
                    "player_name": None,
                    "player_order": None,
                    "goals": 0,
                    "assists": 0,
                    "own_goals": 0,
                    "yellow_cards": 0,
                    "red_cards": 0,
                    "scraped_at_utc": scraped_at,
                }
            )

    scraped_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    for player_name, stats in player_stats.items():
        if sum(stats.values()) > 0:
            rows.append(
                {
                    "match_id": match_ref,
                    "match_url": match_url,
                    "date": date_str,
                    "team": None,
                    "coach": None,
                    "player_name": player_name,
                    "player_order": None,
                    "goals": stats["goals"],
                    "assists": stats["assists"],
                    "own_goals": stats["own_goals"],
                    "yellow_cards": stats["yellow_cards"],
                    "red_cards": stats["red_cards"],
                    "scraped_at_utc": scraped_at,
                }
            )

    return rows


def _parse_retry_after(header_value: Optional[str]) -> Optional[float]:
    if not header_value:
        return None
    try:
        value = float(header_value.strip())
        return max(0.0, value)
    except (TypeError, ValueError):
        return None


async def fetch_match_html_with_backoff(
    client: httpx.AsyncClient,
    match_url: str,
    max_retries: int = 4,
) -> Dict[str, object]:
    """
    Async fetch with exponential backoff + jitter.
    Wait formula per retry: 2^n + jitter (n starts at 1 => ~2s, ~4s, ~8s, ~16s).
    """
    retriable_statuses = {429, 500, 502, 503, 504}

    for attempt in range(max_retries + 1):
        try:
            resp = await client.get(match_url)
            status_code = resp.status_code

            retry_after = _parse_retry_after(resp.headers.get("Retry-After"))
            rate_limit_limit = resp.headers.get("X-RateLimit-Limit")

            if status_code == 200:
                return {
                    "ok": True,
                    "html": resp.text,
                    "status_code": 200,
                    "error": None,
                    "retry_after": retry_after,
                    "rate_limit_limit": rate_limit_limit,
                }

            if status_code in retriable_statuses and attempt < max_retries:
                base_wait = float(2 ** (attempt + 1))
                jitter = random.uniform(0.0, 1.0)
                wait_seconds = (retry_after if (status_code == 429 and retry_after is not None) else base_wait) + jitter
                print(
                    f"[RETRY] HTTP {status_code} for {match_url}. "
                    f"Attempt {attempt + 1}/{max_retries}. Sleeping {wait_seconds:.2f}s..."
                )
                await asyncio.sleep(wait_seconds)
                continue

            return {
                "ok": False,
                "html": None,
                "status_code": status_code,
                "error": f"HTTP {status_code}",
                "retry_after": retry_after,
                "rate_limit_limit": rate_limit_limit,
            }

        except httpx.RequestError as exc:
            if attempt < max_retries:
                base_wait = float(2 ** (attempt + 1))
                jitter = random.uniform(0.0, 1.0)
                wait_seconds = base_wait + jitter
                print(
                    f"[RETRY] Request error for {match_url}: {exc}. "
                    f"Attempt {attempt + 1}/{max_retries}. Sleeping {wait_seconds:.2f}s..."
                )
                await asyncio.sleep(wait_seconds)
                continue
            return {
                "ok": False,
                "html": None,
                "status_code": None,
                "error": f"REQUEST_ERROR: {exc}",
                "retry_after": None,
                "rate_limit_limit": None,
            }

    return {
        "ok": False,
        "html": None,
        "status_code": None,
        "error": "UNKNOWN_ERROR",
        "retry_after": None,
        "rate_limit_limit": None,
    }


async def discover_concurrency_ceiling(
    client: httpx.AsyncClient,
    pending_match_refs: List[str],
    start_concurrency: int = 5,
    max_concurrency: int = 30,
    probe_requests: int = 40,
) -> Tuple[int, Dict[str, Optional[object]]]:
    """
    Step A/B/C:
    - Start at low concurrency (default 5) with no artificial baseline sleep.
    - If a probe round contains any 200, increase concurrency by 1.
    - If any 429 appears, treat it as ceiling discovered.
    - Capture Retry-After / X-RateLimit-Limit headers if present.
    """
    if not pending_match_refs:
        return start_concurrency, {
            "ceiling_found": False,
            "retry_after": None,
            "x_rate_limit_limit": None,
            "probe_rounds": 0,
        }

    probe_refs = pending_match_refs[: min(len(pending_match_refs), probe_requests)]
    current = max(1, start_concurrency)
    max_safe = max(1, start_concurrency)
    probe_rounds = 0
    idx = 0

    seen_retry_after: Optional[float] = None
    seen_rate_limit_limit: Optional[str] = None

    while idx < len(probe_refs) and current <= max_concurrency:
        batch_refs = probe_refs[idx : idx + current]
        if not batch_refs:
            break

        probe_rounds += 1
        tasks = [
            fetch_match_html_with_backoff(
                client=client,
                match_url=f"{BASE_URL}/matches/report/{match_ref}",
                max_retries=0,
            )
            for match_ref in batch_refs
        ]
        results = await asyncio.gather(*tasks)
        idx += len(batch_refs)

        saw_200 = False
        saw_429 = False
        for result in results:
            if result.get("status_code") == 200:
                saw_200 = True
            if result.get("status_code") == 429:
                saw_429 = True
            if seen_retry_after is None and result.get("retry_after") is not None:
                seen_retry_after = float(result["retry_after"])
            if seen_rate_limit_limit is None and result.get("rate_limit_limit"):
                seen_rate_limit_limit = str(result["rate_limit_limit"])

        if saw_429:
            return max(1, current - 1), {
                "ceiling_found": True,
                "retry_after": seen_retry_after,
                "x_rate_limit_limit": seen_rate_limit_limit,
                "probe_rounds": probe_rounds,
            }

        if saw_200:
            max_safe = max(max_safe, current)
            current += 1
        else:
            break

    return min(max_safe, max_concurrency), {
        "ceiling_found": False,
        "retry_after": seen_retry_after,
        "x_rate_limit_limit": seen_rate_limit_limit,
        "probe_rounds": probe_rounds,
    }


def _flatten_match_refs(match_ids_by_country_year: dict) -> List[str]:
    seen: Set[str] = set()
    match_refs: List[str] = []

    for key in sorted(match_ids_by_country_year.keys()):
        value = match_ids_by_country_year[key]
        for match_ref in sorted(str(item) for item in value):
            if match_ref not in seen:
                seen.add(match_ref)
                match_refs.append(match_ref)

    return match_refs


def _load_match_refs_from_cell4(match_ids_by_country_year: Optional[dict] = None) -> List[str]:
    source_mapping = match_ids_by_country_year or globals().get("match_ids_by_country_year")
    if not source_mapping:
        raise ValueError("Run Cell 4 first so match_ids_by_country_year is available.")
    match_refs = _flatten_match_refs(source_mapping)
    if not match_refs:
        raise ValueError("Cell 4 did not provide any match refs.")
    return match_refs


def _load_existing_scraped_match_refs(output_csv_path: Path) -> Set[str]:
    if not output_csv_path.exists():
        return set()
    existing_df = pd.read_csv(output_csv_path, low_memory=False)
    if existing_df.empty or "match_id" not in existing_df.columns:
        return set()
    
    # Missing crucial newly added fields means it wasn't fully scraped under the latest script.
    # Only keep match IDs that have valid data in 'goals' (can be 0 or 0.0 but not NaN).
    if "goals" in existing_df.columns:
        valid_df = existing_df.dropna(subset=["goals"])
        return set(valid_df["match_id"].dropna().astype(str).tolist())
        
    return set(existing_df["match_id"].dropna().astype(str).tolist())


async def scrape_all_match_lineups_and_coaches(
    output_csv_path: Path,
    run_scraper: bool = False,
    force_rescrape: bool = False,
    match_ids_by_country_year: Optional[dict] = None,
    user_agent: str = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    min_sleep_seconds: float = 0.0,
    max_sleep_seconds: float = 0.0,
    long_break_every: int = 40,
    long_break_seconds: int = 60,
    checkpoint_every: int = 25,
    request_timeout: int = 30,
    start_concurrency: int = 5,
    max_concurrency: int = 30,
):
    """Async bulk scraper that consumes the match refs produced by Cell 4."""
    if not run_scraper:
        print("Safety stop: set run_scraper=True to execute bulk scraping.")
        return None

    output_csv_path.parent.mkdir(parents=True, exist_ok=True)

    all_match_refs = _load_match_refs_from_cell4(match_ids_by_country_year)

    existing_scraped_refs = _load_existing_scraped_match_refs(output_csv_path)
    if force_rescrape:
        pending_match_refs = all_match_refs
    else:
        pending_match_refs = [match_ref for match_ref in all_match_refs if match_ref not in existing_scraped_refs]

    print(f"Total match refs in input: {len(all_match_refs)}")
    print(f"Already scraped refs in output: {len(existing_scraped_refs)}")
    print(f"Pending refs for this run: {len(pending_match_refs)}")

    if not pending_match_refs:
        print("No pending matches. Output is already up to date.")
        return {
            "stats": {
                "total_input_refs": len(all_match_refs),
                "already_scraped_refs": len(existing_scraped_refs),
                "pending_refs": 0,
                "processed": 0,
                "success": 0,
                "failed": 0,
            },
            "new_rows": pd.DataFrame(),
        }

    timeout = httpx.Timeout(request_timeout)
    limits = httpx.Limits(max_connections=max_concurrency * 2, max_keepalive_connections=max_concurrency)

    new_rows: List[Dict[str, object]] = []
    failed_rows: List[Dict[str, object]] = []

    processed = 0
    success = 0

    discovered_retry_after: Optional[float] = None
    discovered_rate_limit_limit: Optional[str] = None
    rate_limit_ceiling: Optional[int] = None

    async with httpx.AsyncClient(timeout=timeout, headers={"User-Agent": user_agent}, limits=limits, follow_redirects=True) as client:
        adaptive_concurrency, probe_info = await discover_concurrency_ceiling(
            client=client,
            pending_match_refs=pending_match_refs,
            start_concurrency=start_concurrency,
            max_concurrency=max_concurrency,
            probe_requests=40,
        )

        if probe_info.get("retry_after") is not None:
            discovered_retry_after = float(probe_info["retry_after"])
        if probe_info.get("x_rate_limit_limit") is not None:
            discovered_rate_limit_limit = str(probe_info["x_rate_limit_limit"])
        if probe_info.get("ceiling_found"):
            rate_limit_ceiling = adaptive_concurrency

        print(
            f"[ADAPT] Starting async scrape with concurrency={adaptive_concurrency} "
            f"(ceiling_found={probe_info.get('ceiling_found')}, probe_rounds={probe_info.get('probe_rounds')})"
        )
        if discovered_rate_limit_limit:
            print(f"[HEADER] X-RateLimit-Limit: {discovered_rate_limit_limit}")
        if discovered_retry_after is not None:
            print(f"[HEADER] Retry-After hint: {discovered_retry_after}s")

        index = 0
        while index < len(pending_match_refs):
            batch_refs = pending_match_refs[index : index + adaptive_concurrency]
            if not batch_refs:
                break

            tasks = [
                fetch_match_html_with_backoff(
                    client=client,
                    match_url=f"{BASE_URL}/matches/report/{match_ref}",
                    max_retries=4,
                )
                for match_ref in batch_refs
            ]
            batch_results = await asyncio.gather(*tasks)
            index += len(batch_refs)

            saw_200_in_batch = False
            saw_429_in_batch = False

            for match_ref, result in zip(batch_refs, batch_results):
                match_url = f"{BASE_URL}/matches/report/{match_ref}"
                processed += 1

                if result.get("retry_after") is not None and discovered_retry_after is None:
                    discovered_retry_after = float(result["retry_after"])
                if result.get("rate_limit_limit") and discovered_rate_limit_limit is None:
                    discovered_rate_limit_limit = str(result["rate_limit_limit"])

                status_code = result.get("status_code")
                if status_code == 429:
                    saw_429_in_batch = True
                if status_code == 200:
                    saw_200_in_batch = True

                if result.get("ok") and result.get("html") is not None:
                    rows = extract_match_lineups_and_coaches_from_html(match_ref, match_url, str(result["html"]))
                    if rows:
                        new_rows.extend(rows)
                    else:
                        # Mark as scraped even if team boxes are missing.
                        new_rows.append(
                            {
                                "match_id": match_ref,
                                "match_url": match_url,
                                "date": None,
                                "team": None,
                                "coach": None,
                                "player_name": None,
                                "player_order": None,
                                "goals": 0,
                                "assists": 0,
                                "own_goals": 0,
                                "yellow_cards": 0,
                                "red_cards": 0,
                                "scraped_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                            }
                        )
                    success += 1
                else:
                    failed_rows.append(
                        {
                            "match_id": match_ref,
                            "match_url": match_url,
                            "error": result.get("error"),
                            "status_code": status_code,
                            "failed_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                        }
                    )

                # Incremental checkpoint so a crash does not lose progress.
                if processed % checkpoint_every == 0 and new_rows:
                    batch_df = pd.DataFrame(new_rows)
                    if output_csv_path.exists():
                        existing_df = pd.read_csv(output_csv_path, low_memory=False)
                        combined_df = pd.concat([existing_df, batch_df], ignore_index=True)
                    else:
                        combined_df = batch_df

                    combined_df = combined_df.drop_duplicates(
                        subset=["match_id", "team", "player_name", "player_order"],
                        keep="last",
                    )
                    combined_df.to_csv(output_csv_path, index=False, encoding="utf-8")
                    print(f"[CHECKPOINT] Saved progress after {processed} matches -> {output_csv_path}")
                    new_rows = []

            if saw_429_in_batch:
                rate_limit_ceiling = max(1, min(rate_limit_ceiling or adaptive_concurrency, adaptive_concurrency - 1))
                adaptive_concurrency = rate_limit_ceiling
                print(f"[ADAPT] 429 detected. Reducing concurrency to {adaptive_concurrency}.")
            elif saw_200_in_batch and rate_limit_ceiling is None and adaptive_concurrency < max_concurrency:
                adaptive_concurrency += 1
                print(f"[ADAPT] Batch successful. Increasing concurrency to {adaptive_concurrency}.")

            if processed % long_break_every == 0:
                print(f"[PAUSE] Processed {processed} matches. Cooling down for {long_break_seconds}s...")
                await asyncio.sleep(long_break_seconds)
            elif max_sleep_seconds > 0:
                await asyncio.sleep(random.uniform(min_sleep_seconds, max_sleep_seconds))

    # Final write for any unsaved rows.
    if new_rows:
        batch_df = pd.DataFrame(new_rows)
        if output_csv_path.exists():
            existing_df = pd.read_csv(output_csv_path, low_memory=False)
            combined_df = pd.concat([existing_df, batch_df], ignore_index=True)
        else:
            combined_df = batch_df

        combined_df = combined_df.drop_duplicates(
            subset=["match_id", "team", "player_name", "player_order"],
            keep="last",
        )
        combined_df.to_csv(output_csv_path, index=False, encoding="utf-8")

    # Save failures for targeted reruns.
    failed_csv_path = output_csv_path.with_name("afc_match_lineups_coaches_failed.csv")
    if failed_rows:
        pd.DataFrame(failed_rows).to_csv(failed_csv_path, index=False, encoding="utf-8")

    stats = {
        "total_input_refs": len(all_match_refs),
        "already_scraped_refs": len(existing_scraped_refs),
        "pending_refs": len(pending_match_refs),
        "processed": processed,
        "success": success,
        "failed": len(failed_rows),
        "output_csv": str(output_csv_path),
        "failed_csv": str(failed_csv_path) if failed_rows else None,
        "completed_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "start_concurrency": start_concurrency,
        "max_concurrency": max_concurrency,
        "final_concurrency": adaptive_concurrency,
        "rate_limit_ceiling": rate_limit_ceiling,
        "retry_after_hint": discovered_retry_after,
        "x_rate_limit_limit": discovered_rate_limit_limit,
    }

    print("Run stats:", stats)
    return {
        "stats": stats,
        "new_rows": pd.DataFrame(new_rows),
        "failed_rows": pd.DataFrame(failed_rows),
    }


# Paths
lineups_output_csv = Path("downloaded_files/scrape_exports/afc_match_lineups_coaches.csv")

# Safety switches
RUN_LINEUP_COACH_BULK_SCRAPER = False  # Change to True when you intentionally want to run.
FORCE_RE_SCRAPE_ALL = False        # Keep False to resume/skip already scraped matches.

bulk_lineup_result = await scrape_all_match_lineups_and_coaches(
    output_csv_path=lineups_output_csv,
    run_scraper=RUN_LINEUP_COACH_BULK_SCRAPER,
    force_rescrape=FORCE_RE_SCRAPE_ALL,
    match_ids_by_country_year=match_ids_by_country_year,
    min_sleep_seconds=0.0,
    max_sleep_seconds=0.0,
    long_break_every=40,
    long_break_seconds=60,
    checkpoint_every=25,
    request_timeout=30,
    start_concurrency=5,
    max_concurrency=30,
)

bulk_lineup_result

Safety stop: set run_scraper=True to execute bulk scraping.


In [ ]:
# Load existing scraped data and find unique players
import pandas as pd
from pathlib import Path

lineups_output_csv = Path("../downloaded_files/scrape_exports/afc_match_lineups_coaches.csv")

if lineups_output_csv.exists():
    df_lineups = pd.read_csv(lineups_output_csv, low_memory=False)
    print(f"Loaded {len(df_lineups)} rows from {lineups_output_csv}")
    
    # Check if 'player_link' column exists. If not, maybe use 'player_name' or similar if intended.
    # The user specifically asked for 'player_link'.
    if 'player_link' in df_lineups.columns:
        unique_player_links = df_lineups['player_link'].dropna().unique()
        print(f"Total unique player links found: {len(unique_player_links)}")
        # Display some
        print("Sample links:", unique_player_links[:5])
    else:
        print("Column 'player_link' not found in the CSV.")
        print("Available columns:", df_lineups.columns.tolist())
else:
    print(f"File {lineups_output_csv} does not exist yet. Run the bulk scraper first.")


Loaded 209936 rows from downloaded_files/scrape_exports/afc_match_lineups_coaches.csv
Total unique player links found: 25281
Sample links: ['https://national-football-teams.com/player/3656/Zamirbek_Jumagulov.html'
 'https://national-football-teams.com/player/3659/Vyacheslav_Pryanishnikov.html'
 'https://national-football-teams.com/player/3663/Vyacheslav_Amin.html'
 'https://national-football-teams.com/player/3664/Vladimir_Salo.html'
 'https://national-football-teams.com/player/3667/Zakir_Jalilov.html']


In [ ]:
import re
from bs4 import BeautifulSoup

# --- Tier Hierarchy Dicts ---
# Modify these lists as appropriate for the historical anchor
TIER_1_EURO = {"England", "Spain", "Germany", "Italy", "France"}
TIER_2_EURO_6_15 = {"Portugal", "Netherlands", "Belgium", "Scotland", "Turkey", "Russia", "Austria", "Switzerland", "Ukraine", "Greece"}
TIER_2_AFC = {"Japan", "Saudi_Arabia", "China"}
TIER_2_SA = {"Brazil", "Argentina"}

TIER_3_EURO_16_30 = {"Croatia", "Serbia", "Czech_Republic", "Sweden", "Denmark", "Poland", "Norway", "Israel", "Cyprus", "Romania"}
TIER_3_AFC = {"South_Korea", "United_Arab_Emirates", "Thailand", "Qatar"}

TIER_4_AFC = {"Australia", "Uzbekistan", "Iran", "Iraq", "Jordan", "Syria"}
# Other Euro/SA Tier 1 defaults to Tier 4

def get_league_tier(country: str, division_str: str) -> int:
    """Map country and division (e.g. 'I', 'II') to the 1-6 scale."""
    if not division_str:
        return 6
    
    div = 6
    if division_str == 'I': div = 1
    elif division_str == 'II': div = 2
    elif division_str == 'III': div = 3
    elif division_str == 'IV': div = 4
    elif division_str == 'V': div = 5
    
    if div == 1:
        if country in TIER_1_EURO: return 1
        if country in TIER_2_EURO_6_15 or country in TIER_2_AFC or country in TIER_2_SA: return 2
        if country in TIER_3_EURO_16_30 or country in TIER_3_AFC: return 3
        
        # Consider remaining big confederations as Tier 4, others 5/6. 
        # Example for fallback (You can expand this):
        if country in TIER_4_AFC: return 4
        return 5 # Regional defaults (Vietnam, Malaysia, etc.)
    elif div == 2:
        if country in TIER_1_EURO: return 3 # 2nd tier of Top 5
        if country in TIER_2_EURO_6_15 or country in TIER_2_AFC or country in TIER_2_SA: return 5 # 2nd tier of Mid Euro -> Tier 5
        return 6 # Lower than that -> developing
    else:
        return 6 # Division 3+ defaults to local level


def parse_player_html(html_str: str) -> dict:
    soup = BeautifulSoup(html_str, 'html.parser')
    
    # 1) Characteristics
    pos_elem = soup.find(attrs={"itemprop": "positionName"})
    position = pos_elem.text.strip() if pos_elem else None
    
    birth_elem = soup.find(attrs={"itemprop": "birthDate"})
    birth_year = None
    if birth_elem:
        birth_year_match = re.search(r'\d{4}', birth_elem.text)
        if birth_year_match:
            birth_year = int(birth_year_match.group(0))
            
    height_elem = soup.find(attrs={"itemprop": "height"})
    height = height_elem.text.strip() if height_elem else None

    # 2) Career Peak
    peak_tier = 6 # Default lowest
    matches_in_peak = 0
    best_league_info = None

    # Find the career stats table (table.clubs)
    # The page uses DataTables which clones the table header, so we look for the table with a tbody
    for stats_table in soup.find_all('table', class_='clubs'):
        tbody = stats_table.find('tbody')
        if not tbody:
            continue
            
        for row in tbody.find_all('tr'):
            # Extract matches
            matches_td = row.find('td', class_='stats matches')
            if not matches_td: continue
            matches_text = matches_td.text.strip()
            if not matches_text.isdigit(): continue
            matches = int(matches_text)
            
            # We only consider leagues where they played >= 10 matches
            if matches < 10:
                continue
            
            # Get Country Name
            country_td = row.find('td', class_='country')
            if not country_td: continue
            country_a = country_td.find('a', href=re.compile(r'/country/'))
            if not country_a: continue
            
            country_href = country_a.get('href', '')
            # e.g., /country/204/2025/Vietnam.html
            country = country_href.split('/')[-1].replace('.html', '')

            # Get Division level (e.g. "I", "II")
            rank_td = row.find('td', class_='stats rank')
            division = None
            if rank_td:
                abbr = rank_td.find('abbr')
                if abbr:
                    division = abbr.text.strip()
            
            # Calculate tier
            tier = get_league_tier(country, division)
            
            if tier < peak_tier:
                peak_tier = tier
                matches_in_peak = matches
                best_league_info = (country, division)
            elif tier == peak_tier:
                matches_in_peak += matches
                if not best_league_info:
                    best_league_info = (country, division)
                    
    return {
        "position": position,
        "birth_year": birth_year,
        "height": height,
        "peak_tier": peak_tier,
        "peak_matches": matches_in_peak,
        "best_league_info": best_league_info
    }

# Quick test with local HTML files
if __name__ == "__main__":
    import os
    if os.path.exists("../test.html") and os.path.exists("../test2.html"):
        # Since test.html and test2.html contain different parts of player page,
        # we will join them for the test parse
        with open("../test.html", "r") as f:
            html1 = f.read()
        with open("../test2.html", "r") as f:
            html2 = f.read()
        
        combined_html = html1 + html2
        result = parse_player_html(combined_html)
        print("Scrape Result:", result)


Scrape Result: {'position': 'Attacking Midfielder', 'birth_year': 1997, 'height': '1.66m', 'peak_tier': 3, 'peak_matches': 12, 'best_league_info': ('France', 'II')}


In [ ]:
import asyncio
import random
from pathlib import Path
from datetime import datetime, timezone
import httpx
import pandas as pd

# Paths
players_output_csv = Path("../downloaded_files/scrape_exports/afc_players_data.csv")
players_failed_csv = Path("../downloaded_files/scrape_exports/afc_players_data_failed.csv")

# Config
RUN_PLAYER_SCRAPER = False
FORCE_RE_SCRAPE_PLAYERS = False
PLAYER_CONCURRENCY = 10
CHECKPOINT_EVERY = 50

# Reuse backoff helper
async def fetch_player_html_with_backoff(client: httpx.AsyncClient, url: str, max_retries: int = 4):
    retriable_statuses = {429, 500, 502, 503, 504}
    for attempt in range(max_retries + 1):
        try:
            resp = await client.get(url)
            status_code = resp.status_code

            if status_code == 200:
                return {"ok": True, "html": resp.text, "status_code": 200, "error": None}

            if status_code in retriable_statuses and attempt < max_retries:
                # Basic exponential backoff + jitter
                wait_seconds = float(2 ** (attempt + 1)) + random.uniform(0.5, 1.5)
                # Parse Retry-After if available
                retry_after = resp.headers.get("Retry-After")
                if retry_after and retry_after.isdigit():
                    wait_seconds = max(wait_seconds, float(retry_after))
                
                print(f"[RETRY] HTTP {status_code} for {url}. Attempt {attempt + 1}. Sleep {wait_seconds:.2f}s")
                await asyncio.sleep(wait_seconds)
                continue

            return {"ok": False, "html": None, "status_code": status_code, "error": f"HTTP {status_code}"}

        except httpx.RequestError as exc:
            if attempt < max_retries:
                wait_seconds = float(2 ** (attempt + 1)) + random.uniform(0.5, 1.5)
                await asyncio.sleep(wait_seconds)
                continue
            return {"ok": False, "html": None, "status_code": None, "error": f"REQUEST_ERROR: {exc}"}

    return {"ok": False, "html": None, "status_code": None, "error": "UNKNOWN"}


async def scrape_all_players(
    links: list,
    output_csv: Path,
    failed_csv: Path,
    run_scraper: bool = False,
    force_rescrape: bool = False,
    concurrency: int = 10,
    checkpoint_every: int = 50
):
    if not run_scraper:
        print("Safety stop: set RUN_PLAYER_SCRAPER=True to run.")
        return
        
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    
    # Check existing
    existing_links = set()
    if not force_rescrape and output_csv.exists():
        df_existing = pd.read_csv(output_csv, low_memory=False)
        if "player_link" in df_existing.columns:
            existing_links = set(df_existing["player_link"].dropna())
            
    pending_links = [l for l in links if l not in existing_links]
    print(f"Total target players: {len(links)}")
    print(f"Already scraped: {len(existing_links)}")
    print(f"Pending scrape: {len(pending_links)}")
    
    if not pending_links:
        return
        
    limits = httpx.Limits(max_connections=concurrency * 2, max_keepalive_connections=concurrency)
    timeout = httpx.Timeout(30.0)
    
    new_rows = []
    failed_rows = []
    processed = 0
    
    async with httpx.AsyncClient(headers={"User-Agent": "Mozilla/5.0"}, limits=limits, timeout=timeout, follow_redirects=True) as client:
        index = 0
        while index < len(pending_links):
            batch = pending_links[index : index + concurrency]
            index += len(batch)
            
            tasks = []
            for player_link in batch:
                # Ensure the link is absolute
                url = player_link if player_link.startswith("http") else f"{BASE_URL}{player_link}"
                tasks.append(fetch_player_html_with_backoff(client, url))
                
            results = await asyncio.gather(*tasks)
            
            for player_link, result in zip(batch, results):
                processed += 1
                try:
                    if result["ok"]:
                        parsed_data = parse_player_html(result["html"])
                        parsed_data["player_link"] = player_link
                        parsed_data["scraped_at_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
                        new_rows.append(parsed_data)
                    else:
                        failed_rows.append({
                            "player_link": player_link,
                            "error": result["error"],
                            "status_code": result["status_code"]
                        })
                except Exception as e:
                    failed_rows.append({
                        "player_link": player_link,
                        "error": f"PARSE_ERROR: {e}",
                        "status_code": result.get("status_code")
                    })
                    
            # Checkpoint
            if processed % checkpoint_every == 0 and new_rows:
                df_batch = pd.DataFrame(new_rows)
                if output_csv.exists():
                    df_all = pd.concat([pd.read_csv(output_csv, low_memory=False), df_batch], ignore_index=True)
                else:
                    df_all = df_batch
                    
                df_all.drop_duplicates(subset=["player_link"], keep="last", inplace=True)
                df_all.to_csv(output_csv, index=False)
                new_rows = []
                print(f"[CHECKPOINT] Saved {processed}/{len(pending_links)} players...")
                
            # Add a small delay between batches
            await asyncio.sleep(random.uniform(0.5, 1.5))
            
    # Final save
    if new_rows:
        df_batch = pd.DataFrame(new_rows)
        if output_csv.exists():
            df_all = pd.concat([pd.read_csv(output_csv, low_memory=False), df_batch], ignore_index=True)
        else:
            df_all = df_batch
        df_all.drop_duplicates(subset=["player_link"], keep="last", inplace=True)
        df_all.to_csv(output_csv, index=False)
        print(f"[FINISHED] Saved final remaining players.")
        
    if failed_rows:
        df_failed = pd.DataFrame(failed_rows)
        if failed_csv.exists():
            df_all_failed = pd.concat([pd.read_csv(failed_csv, low_memory=False), df_failed], ignore_index=True)
        else:
            df_all_failed = df_failed
        df_all_failed.drop_duplicates(subset=["player_link"], keep="last", inplace=True)
        df_all_failed.to_csv(failed_csv, index=False)
        print(f"[NOTE] Logged {len(failed_rows)} failed requests to {failed_csv}")

# Check if unique_player_links exists, if not, try to use it if previously extracted from cell 10
if 'unique_player_links' in globals():
    await scrape_all_players(
        links=list(unique_player_links),
        output_csv=players_output_csv,
        failed_csv=players_failed_csv,
        run_scraper=RUN_PLAYER_SCRAPER,
        force_rescrape=FORCE_RE_SCRAPE_PLAYERS,
        concurrency=PLAYER_CONCURRENCY,
        checkpoint_every=CHECKPOINT_EVERY
    )
else:
    print("Variable 'unique_player_links' not found. Ensure Cell 10 was executed successfully.")


Safety stop: set RUN_PLAYER_SCRAPER=True to run.
